# Example notebook make n(z) with KNN

This will run the KNN algorithm to get p(z) for all objects in the WFD sample,
then it will use the mode of the p(z) to get the tomographic bin assignments and stack the p(z) to get n(z)

#### Standard imports and rail catalog setup

In [ ]:
import tables_io, qp
import numpy as np
import os
import matplotlib.pyplot as plt
from rail.estimation.algos.k_nearneigh import KNearNeighEstimator, KNearNeighInformer
from rail.estimation.algos.naive_stack import NaiveStackMaskedSummarizer
from rail.estimation.algos.uniform_binning import UniformBinningClassifier

from rail.core.data import TableHandle, QPHandle
from rail.utils import catalog_utils

from nz_data_challenge.utils import TOMO_BIN_EDGES, TASKSETS, SIMS, SCENARIOS

# RAIL setup
catalog_utils.clear()
catalog_utils.load_yaml("../tests/catalogs.yaml")
CATALOG_TAG = "cardinal_roman_rubin"
catalog_utils.apply(CATALOG_TAG)

#### Paths for nz challenge related files

In [ ]:
submission_name = "rail_knn_4tasks"
test_only = True

if test_only:
    submit_dir = f'../submission_test/{submission_name}'
    test_suffix = 'ddf_00'
    model_dir = f'../models_test/{submission_name}'
else:
    submit_dir = f'../submission/{submission_name}'
    test_suffix = 'wfd'
    model_dir = f'../models/{submission_name}'

public_dir = '../public'
try:
    os.makedirs(submit_dir)
except:
    pass

### Loop over everything and do everything

In [ ]:
do_pz_estimate = True
do_nz_bin_assignment = True
do_nz_estimate = True

# taskset = ['taskset_1', 'taskset_2']
# sims = ['cardinal', 'flagship']
# scenarios = ['1yr', '4yr']

# loop over tasksets, simulations and scenarios
for taskset in TASKSETS:

    # Get the desired bining for this taskset
    tomo_bin_edges = TOMO_BIN_EDGES[taskset]
    tomo_bin_centers = 0.5*(tomo_bin_edges[0:-1]+tomo_bin_edges[1:])
    n_tomo_bins = len(tomo_bin_edges) - 1
    
    for sim in SIMS:
        for scenario in SCENARIOS:
            
            # Make the file names for this particular setup
            wfd_file = f"{public_dir}/nz_challenge_{taskset}_{sim}_{scenario}_{test_suffix}.hdf5"
            model_file = f"{model_dir}/pz_challenge_{taskset}_{sim}_pz_model_1yr.pkl"
            pz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_pz_estimate_{test_suffix}.hdf5"
            nz_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_estimate_{test_suffix}.hdf5"
            nz_sample_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_nz_samples_{test_suffix}.hdf5"
            bhat_file = f"{submit_dir}/nz_challenge_{taskset}_{sim}_{scenario}_bhat_{test_suffix}.hdf5"
            
            # Make a KNN estiamtor and run it
            knn_estimate = KNearNeighEstimator.make_stage(
                name=f"estimate_{taskset}_{sim}_{scenario}",
                model=model_file,
                hdf5_groupname="",
                id_col='object_id',
                nzbins=301,
                zmax=3.0,
                chunk_size=10000,
                nondetect_val=np.nan,
            )

            # Bin the objects by mode of the p(z) distribution
            bin_classifier = UniformBinningClassifier.make_stage(
                name=f"classify_{taskset}_{sim}_{scenario}",
                zbin_edges=tomo_bin_edges,
                no_assign=-1,
                object_id_col='object_id',
            )

            # Using naive pdf stacking to summarize the n(z) distritubions
            summarizer = NaiveStackMaskedSummarizer.make_stage(
                name=f"summarize_{taskset}_{sim}_{scenario}",
                selected_bin=0,
                n_tomo_bins=n_tomo_bins,
                n_samples=100,
                chunk_size=10000,
            )

            test_handle = TableHandle(f"test_{taskset}_{sim}_{scenario}", path=str(wfd_file))

            if do_pz_estimate:
                pz_estimates = knn_estimate.estimate(test_handle)
                # This is so that the next stage reads the whole file, not just the
                # current chunk               
                pz_estimates.data = None
                os.system(f"cp {pz_estimates.path} {pz_file}")  
            else:
                 pz_estimates = QPHandle(f'output_{taskset}_{sim}_{scenario}', path=pz_file)

            if do_nz_bin_assignment:
                bin_assignments = bin_classifier.classify(pz_estimates)
                # This is so that the next stage reads the whole file, not just the
                # current chunk
                bin_assignments.data = None               
                os.system(f"cp {bin_assignments.path} {bhat_file}")  
            else:
                bin_assignments = TableHandle(f'bhat_{taskset}_{sim}_{scenario}', path=bhat_file)

            if do_nz_estimate:
                samples_nz = summarizer.summarize(pz_estimates, bin_assignments)                
                single_nz = summarizer.get_handle('single_NZ')
                os.system(f"cp {samples_nz.path} {nz_sample_file}")
                os.system(f"cp {single_nz.path} {nz_file}")
            else:
                samples = QPHandle(f'output_summarize', path=nz_sample_file)
                single_nz = QPHandle(f'single_NZ_summarize', path=nz_file)

